# Visualization

> How to visualize camera images published on ROS2 topics using rqt_image_view, rviz2 and Foxglove.

## Why Visualization Matters

You have a camera node publishing images to a ROS2 topic — great. But how do you actually **see** them?

Here are three tools for image visualization, each with different strengths:

| Tool | Best for | Complexity |
|------|----------|------------|
| `rqt_image_view` | Quick debugging, single topic | Very simple |
| `rviz2` | Full 3D context, multiple displays | Medium |
| Foxglove | Modern UI, recording, web-based | Medium |

We'll cover each one focused on **image visualization**. Keep in mind that all three tools can do much more — rviz2 is a full 3D visualization suite, Foxglove is a complete robotics IDE, and rqt has dozens of plugins — but here we focus on seeing camera images.

## rqt_image_view

The simplest way to see an image from a ROS2 topic. One command, one dropdown, done.

### Install

```bash
sudo apt install ros-$ROS_DISTRO-rqt-image-view
```

### Launch

```bash
ros2 run rqt_image_view rqt_image_view
```

A window opens with a dropdown at the top. Select your image topic (e.g. `/camera/image_raw`) and you're done.

You can also launch it from the rqt GUI:

```bash
rqt

# Then: Plugins → Visualization → Image View
```

### Tips

- You can open **multiple instances** to view different topics simultaneously.
- If the topic uses a **compressed transport** (e.g. `/camera/image_raw/compressed`), select that topic directly — rqt_image_view handles `sensor_msgs/Image` and `sensor_msgs/CompressedImage`.
- The **transport hint** dropdown in older versions lets you switch between raw/compressed without changing the topic name.

> **When to use:** Quick sanity check. "Is my camera publishing? What does the image look like?" — that's rqt_image_view territory.

## rviz2

The standard ROS2 visualization tool. It's a full 3D viewer, but for cameras you'll use the **Image** display.

### Install

```bash
sudo apt install ros-$ROS_DISTRO-rviz2
```

### Launch

```bash
rviz2
```

### Add an Image display

1. Click **Add** (bottom left, Displays panel)
2. Go to the **By topic** tab
3. Find your image topic (e.g. `/camera/image_raw`)
4. Select **Image** → OK

The image appears in a panel. You can resize it, undock it, or put it in a tab layout.

### Common issues

**Black screen / no image:**
- Check that the topic actually has data: `ros2 topic hz /camera/image_raw`
- Verify the **QoS** settings match. rviz2 uses `SensorDataQoS` by default — if your publisher uses a different QoS profile, the image won't show. Right-click the display → check the QoS settings.

**"No transform from [camera_frame] to [map]" warning:**
- This is about TF frames. For a standalone camera image you don't need a transform. You can ignore this warning or set the **Fixed Frame** (top of the Displays panel) to `camera_frame` or whatever your node uses.

### Multiple cameras

You can add multiple Image displays, one per topic. Arrange them in panels using **Panels → Add New Panel → rviz_common/Views** or just drag the tabs around.

> **When to use:** You need more than just the image — maybe TF frames, markers, point clouds — or you're already in rviz2 for other reasons.

## Foxglove

A modern, web-based visualization platform for robotics. Think of it as "rviz2 but with a nicer UI and built-in recording."

To connect Foxglove to your ROS2 system, you need the **bridge** — a native ROS2 node that exposes your system over WebSocket.

### Foxglove Bridge (recommended)

The bridge is a native ROS2 node that exposes your system over WebSocket.

#### Install

```bash
sudo apt install ros-$ROS_DISTRO-foxglove-bridge
```

You also need the Foxglove app. Download it from [foxglove.dev](https://foxglove.dev/download) — available as AppImage, `.deb`, or snap.

#### Launch

**Terminal 1** — start the bridge:

```bash
ros2 launch foxglove_bridge foxglove_bridge_launch.xml
```

By default it listens on `ws://localhost:8765`. You can change the port:

```bash
ros2 launch foxglove_bridge foxglove_bridge_launch.xml port:=9999
```

**Terminal 2** — open the Foxglove app:

```bash
foxglove
```

1. Select **Open connection** → **Foxglove WebSocket**
2. URL should be `ws://localhost:8765` (or your custom port)
3. Click **Open**

#### View an image

1. Click **Add panel** (top left)
2. Select **Image**
3. In the panel settings (gear icon), choose your topic (e.g. `/camera/image_raw`)

You can add multiple Image panels, resize them, and save the layout for later.

## Comparison

| Feature | rqt_image_view | rviz2 | Foxglove |
|---------|----------------|-------|----------|
| Install size | Small | Medium | Medium (app + bridge) |
| Ease of use | ★★★★★ | ★★★ | ★★★★ |
| Multiple images | Multiple windows | Multiple displays | Multiple panels |
| Recording | No | No | Yes (built-in) |
| 3D context | No | Yes | Yes |
| Web-based | No | No | Yes |
| Best for | Quick debug | Full ROS2 viz | Modern workflow |

### Which one should I use?

- **Just checking if my camera works?** → `rqt_image_view`. Fastest path.
- **Working with TF, point clouds, or other 3D data too?** → `rviz2`. It's the standard.
- **Want a modern UI, layouts, and recording?** → Foxglove with the bridge.

In practice, you'll probably end up using all three at different times. That's fine.

## Troubleshooting

### No image appears

1. **Is data actually flowing?**

   ```bash
   ros2 topic hz /camera/image_raw
   ros2 topic echo /camera/image_raw --once  # should show header + data
   ```

2. **Is the message type correct?**

   ```bash
   ros2 topic info /camera/image_raw
   # Should show sensor_msgs/msg/Image or sensor_msgs/msg/CompressedImage
   ```

### Black screen

- **QoS mismatch** — the most common cause. The subscriber (viewer) and publisher must agree on QoS. Check your publisher's QoS profile and match it in the viewer.
- **Wrong encoding** — if your node publishes `mono8` but the viewer expects `bgr8`, you'll get garbage or a black screen. Check the `encoding` field in the message.

### Foxglove shows "No data received"

- Make sure the bridge is running: `ros2 node list | grep foxglove`
- Check the WebSocket URL matches (default: `ws://localhost:8765`)
- If on a different machine, use the IP address and make sure the port is open

### rviz2 TF warnings

- Set **Fixed Frame** to your camera's frame_id (check with `ros2 topic echo /camera/image_raw --once | grep frame_id`)
- Or just ignore the warning — it doesn't affect the Image display

## Next Steps

Now that you can see your camera images, explore more advanced topics:

- **Image transport** (`05_image_transport`) — send compressed images to save bandwidth
- **Calibration** (`06_calibration_and_info`) — fix lens distortion and get accurate measurements
- **Simulation** (`07_simulation`) — test everything without real hardware